# State Conditioned YTD Participation Rescue V1

Purpose: test whether the previous low-drawdown/low-participation profile can be rescued by a stronger trend floor, higher rebound target, warm-start, and return-aware Pareto selection while keeping the 20% drawdown gate intact.

Baseline: `state_conditioned_multi_objective_ytd_gate_v1`. Reference floor: `soxl_backtest_policy_reimplementation_v1`.

In [1]:
from pathlib import Path
import json
import pandas as pd

CARD = Path('runtime/model_selection/card/state_conditioned_ytd_participation_rescue_v1')
if not (CARD / 'summary.json').exists():
    CARD = Path.cwd()
summary = json.loads((CARD / 'summary.json').read_text())
selection = json.loads((CARD / 'selection.json').read_text())
summary['decision'], selection['next_mechanism']

('participation_rescue_improves_return_but_fails_drawdown_no_actor_retention',
 'drawdown_gated_warmstart_checkpoint_selector_before_any_new_actor_retention_attempt')

In [2]:
def replay_metrics(path: Path):
    df = pd.read_csv(path, parse_dates=['datetime'])
    rows = []
    for label, sub in [('full', df), ('ytd_2026', df[df['datetime'] >= '2026-01-02'])]:
        nav = sub['strategy_nav'].astype(float)
        bnh = sub['buyhold_nav'].astype(float)
        ret = (nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0 if len(nav) > 1 else 0.0
        bret = (bnh.iloc[-1] / bnh.iloc[0] - 1.0) * 100.0 if len(bnh) > 1 else 0.0
        dd = ((nav.cummax() - nav) / nav.cummax()).max() * 100.0 if len(nav) > 1 else 0.0
        actions = sub['action'].astype(str)
        rows.append({
            'run': path.stem,
            'window': label,
            'rows': len(sub),
            'start': str(sub['datetime'].iloc[0].date()) if len(sub) else '',
            'end': str(sub['datetime'].iloc[-1].date()) if len(sub) else '',
            'return_pct': ret,
            'buyhold_return_pct': bret,
            'relative_return_pp': ret - bret,
            'max_dd_pct': dd,
            'avg_pos': float(sub['position'].astype(float).mean()) if len(sub) else 0.0,
            'max_pos': float(sub['position'].astype(float).max()) if len(sub) else 0.0,
            'trades': int(actions.isin(['BUY', 'SELL']).sum()) if len(actions) else 0,
        })
    return rows

metrics = pd.DataFrame(
    replay_metrics(CARD / 'smoke_selected_replay.csv') +
    replay_metrics(CARD / 'full_selected_replay.csv')
)
metrics

,run,window,rows,start,end,return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_pos,max_pos,trades
0,smoke_selected_replay,full,1619,2020-01-02,2026-06-11,215.265571,1050.791191,-835.525620,20.118550,0.146340,0.690963,197
1,smoke_selected_replay,ytd_2026,111,2026-01-02,2026-06-11,25.968367,374.153255,-348.184887,7.658116,0.053397,0.392848,19
2,full_selected_replay,full,1619,2020-01-02,2026-06-11,507.527691,1050.791191,-543.263500,40.129281,0.309704,0.871569,275
3,full_selected_replay,ytd_2026,111,2026-01-02,2026-06-11,78.615351,374.153255,-295.537904,15.087274,0.145954,0.627896,21


In [3]:
thresholds = summary['target_thresholds']
gate_table = pd.DataFrame([
    {'gate': 'full_replay_maxdd', 'actual': summary['full_explicit_replay']['full']['max_dd_pct'], 'limit': thresholds['wf_max_drawdown_hard_gate_pct'], 'passed': summary['full_explicit_replay']['full']['max_dd_pct'] < thresholds['wf_max_drawdown_hard_gate_pct']},
    {'gate': 'full_ytd_soxlbacktest_floor', 'actual': summary['full_explicit_replay']['ytd_2026']['return_pct'], 'limit': thresholds['soxlbacktest_reference_ytd_return_pct'], 'passed': summary['full_explicit_replay']['ytd_2026']['return_pct'] >= thresholds['soxlbacktest_reference_ytd_return_pct']},
    {'gate': 'full_ytd_current_best_floor', 'actual': summary['full_explicit_replay']['ytd_2026']['return_pct'], 'limit': thresholds['required_current_best_alignment_ytd_pct'], 'passed': summary['full_explicit_replay']['ytd_2026']['return_pct'] >= thresholds['required_current_best_alignment_ytd_pct']},
    {'gate': 'full_fixed_2022_drawdown', 'actual': summary['full_training']['test_2022_max_dd_pct'], 'limit': thresholds['wf_max_drawdown_hard_gate_pct'], 'passed': summary['full_training']['test_2022_max_dd_pct'] < thresholds['wf_max_drawdown_hard_gate_pct']},
])
gate_table

,gate,actual,limit,passed
0,full_replay_maxdd,40.129281,20.000000,False
1,full_ytd_soxlbacktest_floor,78.615351,132.077978,False
2,full_ytd_current_best_floor,78.615351,213.143462,False
3,full_fixed_2022_drawdown,21.667549,20.000000,False


## Conclusion

- Decision: `participation_rescue_improves_return_but_fails_drawdown_no_actor_retention`.
- Smoke raised full replay return to 215.27%, but full replay MaxDD was still 20.12% and YTD return was only 25.97%.
- Full warm-start raised full replay return to 507.53% and YTD to 78.62%, but full replay MaxDD rose to 40.13% and 2022 fixed-split MaxDD exceeded 20%.
- No actor/checkpoint is retained from this run. Next route is a drawdown-gated warm-start checkpoint selector or an explicit path drawdown cap before another retained-model attempt.